In [21]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib


In [22]:
!git clone https://github.com/tanjumnaher01-spec/Starter-Notebooks-FlyrankAI.git
%cd Starter-Notebooks-FlyrankAI

Cloning into 'Starter-Notebooks-FlyrankAI'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 96 (delta 35), reused 31 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 1.65 MiB | 15.40 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/Starter-Notebooks-FlyrankAI/Starter-Notebooks-FlyrankAI


In [23]:
!find . -name "*.csv"

./data/raw/content_refresh_anonymized.csv
./outputs/refresh_queue_sample.csv


In [24]:
# Cell 1 — setup
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib
import pandas as pd
df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

In [25]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [26]:
# Cell 2 — Signal 1: staleness (check against real refresh flags)
bucket = pd.cut(df["days_since_last_update"], bins=[0,90,180,365,99999])
table1 = df.groupby(bucket)["trend_direction"].value_counts().unstack()
print(table1)
# verdict: look at whether "declining" share rises as staleness increases



trend_direction          down  flat   new  stable    up
days_since_last_update                                 
(0, 90]                 10576   899  2135    3839  3206
(90, 180]                5604   237    76    2099  1155
(180, 365]                 79    15    24      24    27
(365, 99999]                3     1     1       0     0


/tmp/ipykernel_1821/3495640771.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table1 = df.groupby(bucket)["trend_direction"].value_counts().unstack()


In [27]:
# Cell 3 — Signal 2: your second signal (e.g. CTR vs position, or volume)
bucket2 = pd.cut(df["avg_position"], bins=[0,10,20,50,100])
table2 = df.groupby(bucket2)["ctr"].agg(["mean","count"])
print(table2)

                  mean  count
avg_position                 
(0, 10]       0.832373  12983
(10, 20]      0.323443   7273
(20, 50]      0.222345   7225
(50, 100]     0.152525   1299


/tmp/ipykernel_1821/1407502609.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table2 = df.groupby(bucket2)["ctr"].agg(["mean","count"])


In [28]:
# Cell 4 — the rule: score + reason code + action
import os
os.makedirs("work/outputs", exist_ok=True)

def score_row(row):
    s = 0
    reasons = []

    if row["days_since_last_update"] > 365:
        s += 2
        reasons.append("very_stale")
    elif row["days_since_last_update"] > 180:
        s += 1
        reasons.append("stale_content")

    if row["avg_position"] > 50:
        s += 2
        reasons.append("very_low_position")
    elif row["avg_position"] > 20:
        s += 1
        reasons.append("low_position")

    if row["search_volume"] == 0:
        s -= 1
        reasons.append("no_demand")

    reason = reasons[0] if reasons else "none"
    action = "refresh" if s >= 2 else ("monitor" if s >= 1 else "deprioritize")
    return pd.Series([s, reason, action])

df[["score","reason_code","action"]] = df.apply(score_row, axis=1)
queue = df.sort_values("score", ascending=False)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

In [29]:
print(queue.shape)
queue.head()

(30000, 47)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action
15790,content_6476d1d8c050,client_19581e27de,10.0,0.54,MEDIUM,0.85,keyword article,transactional,NaN,NaN,...,0.0,0.0,0.0,moderate,deep,up,31.9,3,stale_content,refresh
7509,content_7a888d3d99c8,client_19581e27de,90.0,0.46,MEDIUM,0.72,keyword article,transactional,NaN,NaN,...,0.0,0.0,0.0,low,deep,down,-100.0,3,stale_content,refresh
26981,content_15fe075b97bc,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1471.0,10414.0,...,0.0,50.0,0.0,low,deep,new,NaN,3,stale_content,refresh
9346,content_d25a099b3726,client_19581e27de,10.0,0.45,MEDIUM,0.03,keyword article,transactional,NaN,NaN,...,0.0,0.0,0.0,low,deep,up,156.6,3,stale_content,refresh
24541,content_d0548e87b05b,client_2c624232cd,NaN,NaN,NaN,NaN,keyword article,commercial,NaN,NaN,...,0.0,25.0,0.0,low,deep,up,47.6,2,very_low_position,refresh


In [30]:
top10 = queue.head(10)
top10[["content_id","score","reason_code","action","days_since_last_update","avg_position","search_volume","trend_direction"]]

,content_id,score,reason_code,action,days_since_last_update,avg_position,search_volume,trend_direction
15790,content_6476d1d8c050,3,stale_content,refresh,313,67.8,10.0,up
7509,content_7a888d3d99c8,3,stale_content,refresh,313,67.6,90.0,down
26981,content_15fe075b97bc,3,stale_content,refresh,304,67.0,NaN,new
9346,content_d25a099b3726,3,stale_content,refresh,305,64.5,10.0,up
24541,content_d0548e87b05b,2,very_low_position,refresh,22,55.5,NaN,up
14362,content_f68b1081eb71,2,very_low_position,refresh,20,54.4,20.0,down
10959,content_19415130d5ee,2,very_low_position,refresh,22,62.1,10.0,down
3840,content_590417dc003b,2,very_low_position,refresh,22,67.9,40.0,stable
5277,content_aa468035de2b,2,very_low_position,refresh,22,61.0,70.0,up
20167,content_013b42b6edf9,2,very_low_position,refresh,22,63.6,NaN,up


#Here's the top-10 review

content_6476d1d8c050 — refresh, score 3 (very stale 313d + very low position 67.8). Would be wrong if: trend_direction is "up" — already recovering, refresh may be unnecessary.
content_7a888d3d99c8 — refresh, score 3 (very stale 313d + very low position 67.6, high volume 90). Would be wrong if: the -100% trend means traffic already hit zero — may need a full rewrite, not just a refresh.
content_15fe075b97bc — refresh, score 3 (very stale 304d + very low position 67.0, trend "new"). Would be wrong if: "new" means not enough data yet — too early to call this a genuine decline.
content_d25a099b3726 — refresh, score 3 (very stale 305d + very low position 64.5). Would be wrong if: trend is "up" already — same recovery risk as row 1.
content_d0548e87b05b — refresh, score 2 (very low position 55.5, only 22 days old). Would be wrong if: content is only 22 days old — position 55 may just reflect normal ranking ramp-up, not a real problem.
content_f68b1081eb71 — refresh, score 2 (very low position 54.4, 20 days old). Would be wrong if: same as above — too new to judge; also trend is "down" but with only 20 days, too little data to be confident.
content_19415130d5ee — refresh, score 2 (very low position 62.1, 22 days old). Would be wrong if: too new (22 days) for the low position to reflect a real problem yet.
content_590417dc003b — refresh, score 2 (very low position 67.9, 22 days old, trend "stable"). Would be wrong if: trend is already "stable" — position may just be settling normally for new content.
content_aa468035de2b — refresh, score 2 (very low position 61.0, 22 days old, volume 70). Would be wrong if: only 22 days old and trend is "up" — likely still climbing, not stuck.
content_013b42b6edf9 — refresh, score 2 (very low position 63.6, 22 days old, trend "up"). Would be wrong if: same — new content trending up, flagging it as a problem is premature.

In [31]:
print(table1)

trend_direction          down  flat   new  stable    up
days_since_last_update                                 
(0, 90]                 10576   899  2135    3839  3206
(90, 180]                5604   237    76    2099  1155
(180, 365]                 79    15    24      24    27
(365, 99999]                3     1     1       0     0


## Section 1: Signal Verification

**Signal 1 — Content Staleness (`days_since_last_update`)**
Linked to: FlyRank's refresh flags.
Bucketed staleness into 4 tiers and compared `trend_direction` counts per tier.

| Bucket | n | % declining |
|---|---|---|
| 0–90 days | 20,655 | 51.2% |
| 90–180 days | 9,171 | 61.1% |
| 180–365 days | 169 | 46.7% |
| 365+ days | 5 | 60.0% |

Verdict: **MIXED** — declining share rises from 51% to 61% between the 0–90 and 90–180 day buckets, supporting the staleness→decline link. But the 180+ buckets have too few rows (n=169, n=5) to draw a reliable conclusion, so the signal only holds confidently in the first two tiers.

In [32]:
print(table2)

                  mean  count
avg_position                 
(0, 10]       0.832373  12983
(10, 20]      0.323443   7273
(20, 50]      0.222345   7225
(50, 100]     0.152525   1299


**Signal 2 — CTR vs Position (`avg_position`, `ctr`)**
Linked to: FlyRank's CTR-fix logic.
Bucketed position into 4 tiers and computed mean CTR per tier.

| Bucket | n | mean CTR |
|---|---|---|
| 0–10 | 12,983 | 0.832 |
| 10–20 | 7,273 | 0.323 |
| 20–50 | 7,225 | 0.222 |
| 50–100 | 1,299 | 0.153 |

Verdict: **CONFIRMED** — mean CTR drops sharply and monotonically as position worsens (0.83 → 0.32 → 0.22 → 0.15), with strong sample sizes across all four buckets. Position is a reliable, well-supported driver of CTR.

## Section 4: Weak Picks

**content_d0548e87b05b** (score 2, very_low_position) — Only 22 days old, position 55.5. This is likely normal ranking ramp-up for new content, not a real decline problem. The rule doesn't account for content age when judging position, which is a gap.

**content_f68b1081eb71** (score 2, very_low_position) — Same issue: 20 days old, position 54.4, trend "down" — but with only 20 days of data, "down" isn't a reliable trend yet.

## Section 5: Self-Check

The rule conflates two different problems under "low position": genuinely stale, declining content, and brand-new content still ramping up in rankings. Adding a minimum content-age threshold (e.g. only flag low position if age > 60 days) would likely fix this. The staleness signal is also less reliable past 180 days due to small sample size in that range.